# Validation Step 2 — Frozen BSF v2 Extraction on OpenBTAI
## Run FROZEN SwinUNETR on 373 unseen scans -> 8466-dim hybrid embeddings

### What this notebook does
1. Load preprocessed OpenBTAI scans (image_4ch + mask_subregions from Step 1)
2. Run FROZEN BSF SwinUNETR — 3 fold checkpoints, NO retraining
3. ROI crop to tumor bbox -> 96^3 -> octant pool -> BSF embedding per scan
4. Average across 3 folds
5. Extract 18-dim shape features from OpenBTAI_Radiomic.xlsx
6. Apply SAME shape_scaler.pkl fitted on Cyprus training — do NOT refit
7. Combine BSF + shape -> 8466-dim hybrid embedding per scan
8. Save openbtai_hybrid_embeddings_v2.npz

### Run on Kaggle GPU (T4 or P100)
Datasets to attach:
- **openbtai-preprocessed**: preprocessed_openbtai/ folder (35GB from Step 1)
- **tavit1-0**: frozen BSF checkpoints + shape_scaler.pkl

### Key decisions
- Frozen model — inference ONLY, no gradient updates
- shape_scaler.pkl from Cyprus training — same scaler, NOT refit on OpenBTAI
- OpenBTAI masks {0,1,3}: WT=mask>0, TC=mask>0 (no edema), ET=mask==3
- t1c intensity stats (features 8-12) set to 0 — not in xlsx


In [14]:
import warnings; warnings.filterwarnings('ignore')
import os, time, json, pickle
import numpy as np
import pandas as pd
import nibabel as nib
import torch
import torch.nn.functional as F
from pathlib import Path
from tqdm import tqdm

try:
    from monai.networks.nets import SwinUNETR
    print('MONAI OK')
except ImportError:
    os.system('pip install -q monai')
    from monai.networks.nets import SwinUNETR
    print('MONAI installed')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'PyTorch: {torch.__version__}')


MONAI OK
Device: cuda
PyTorch: 2.10.0+cu128


In [15]:
# ============================ PATHS ==========================================
IS_KAGGLE = Path('/kaggle/input').exists()
print(f'IS_KAGGLE: {IS_KAGGLE}')

if IS_KAGGLE:
    PREPROCESS_ROOT = None
    for c in [Path('/kaggle/input/datasets/boufafamoamed/openbtai-preprocessed'),
              Path('/kaggle/input/openbtai-preprocessed'),
              Path('/kaggle/input/preprocessed-openbtai-v1')]:
        if c.exists(): PREPROCESS_ROOT = c; break
    if PREPROCESS_ROOT is None:
        for p in Path('/kaggle/input').rglob('openbtai_patient_timelines.csv'):
            PREPROCESS_ROOT = p.parent; break
    assert PREPROCESS_ROOT, 'Cannot find preprocessed OpenBTAI data'

    CKPT_DIR = None
    for p in Path('/kaggle/input').rglob('bsf_fold0_best.pth'):
        CKPT_DIR = p.parent; break
    assert CKPT_DIR, 'Cannot find bsf_fold*_best.pth'

    SCALER_PATH = None
    for p in Path('/kaggle/input').rglob('shape_scaler.pkl'):
        SCALER_PATH = str(p); break

    RADIO_XLSX = None
    for p in Path('/kaggle/input').rglob('OpenBTAI_Radiomic.xlsx'):
        RADIO_XLSX = str(p); break

    MORPH_XLSX = None
    for p in Path('/kaggle/input').rglob('OpenBTAI_MORPHOLOGICAL_MEASUREMENTS.xlsx'):
        MORPH_XLSX = str(p); break

    OUT_DIR = Path('/kaggle/working/openbtai_embeddings')

else:
    PREPROCESS_ROOT = Path('/home/moamed/HDD/validation_data/preprocessed_openbtai')
    CKPT_DIR   = Path('/home/moamed/canada_me/explainable_diseas/implementation_cyprus/Phase3/bsf_fold_outputs/checkpoints')
    SCALER_PATH = '/home/moamed/canada_me/explainable_diseas/implementation_cyprus/Phase3/bsf_fold_outputs/embeddings_v2/shape_scaler.pkl'
    RADIO_XLSX  = '/home/moamed/HDD/validation_data/OpenBTAI_Radiomic.xlsx'
    MORPH_XLSX  = '/home/moamed/HDD/validation_data/OpenBTAI_MORPHOLOGICAL_MEASUREMENTS.xlsx'
    OUT_DIR = Path('/home/moamed/HDD/validation_data/openbtai_embeddings')

OUT_DIR.mkdir(parents=True, exist_ok=True)

# Model config — must exactly match Cyprus Phase3_A1B training
TARGET_SIZE  = (96, 96, 96)
ROI_PADDING  = 16
MODEL_CONFIG = {
    'in_channels': 4, 'out_channels': 3,
    'depths': (2, 2, 2, 2), 'num_heads': (3, 6, 12, 24),
    'feature_size': 48,
    'drop_rate': 0.0, 'attn_drop_rate': 0.0, 'dropout_path_rate': 0.0,
    'use_checkpoint': False,
}
BN_CHANNELS = 768

for p, name in [(PREPROCESS_ROOT, 'PREPROCESS_ROOT'), (CKPT_DIR, 'CKPT_DIR')]:
    print(f'{name}: exists={Path(p).exists()}  {p}')
if PREPROCESS_ROOT and PREPROCESS_ROOT.exists():
    n_gz  = sum(1 for _ in PREPROCESS_ROOT.rglob('image_t1c.nii.gz'))
    n_nii = sum(1 for _ in PREPROCESS_ROOT.rglob('image_t1c.nii'))
    n_scans = max(n_gz, n_nii)
else:
    n_scans = 0
print(f'Preprocessed scans: {n_scans}')
print(f'SCALER_PATH exists: {Path(SCALER_PATH).exists() if SCALER_PATH else False}')


IS_KAGGLE: True
PREPROCESS_ROOT: exists=True  /kaggle/input/datasets/boufafamoamed/openbtai-preprocessed
CKPT_DIR: exists=True  /kaggle/input/datasets/boufafamoamed/openbtai-bsf-assets/checkpoints
Preprocessed scans: 373
SCALER_PATH exists: True


In [16]:
# ============================ LOAD TIMELINES ==================================
tl = pd.read_csv(PREPROCESS_ROOT / 'openbtai_patient_timelines.csv')
scan_index = [(str(r['patient_id']), r['visit_name']) for _, r in tl.iterrows()]
print(f'Timelines: {len(tl)} scans from {tl["patient_id"].nunique()} patients')
print(tl[['patient_id','visit_name','days_since_baseline']].head(6).to_string(index=False))


Timelines: 373 scans from 75 patients
 patient_id visit_name  days_since_baseline
      10005   baseline                    0
      10005        fu1                  115
      10005        fu2                  224
      10005        fu3                  299
      10020   baseline                    0
      10020        fu1                   67


In [17]:
# ============================ MODEL FACTORY ===================================
import inspect

def create_bsf_model():
    kw = dict(**MODEL_CONFIG)
    sig = inspect.signature(SwinUNETR.__init__)
    if 'img_size' in sig.parameters:
        kw['img_size'] = TARGET_SIZE
    return SwinUNETR(**kw)

def load_checkpoint(model, fold):
    ckpt_path = CKPT_DIR / f'bsf_fold{fold}_best.pth'
    sd = torch.load(str(ckpt_path), map_location='cpu', weights_only=False)
    if isinstance(sd, dict):
        for key in ('model_state_dict', 'state_dict', 'model'):
            if key in sd:
                sd = sd[key]; break
    missing, unexpected = model.load_state_dict(sd, strict=False)
    print(f'  Fold {fold}: loaded  missing={len(missing)} unexpected={len(unexpected)}')
    return model

_cache = {}
def _hook(module, inp, out): _cache['bn'] = out
def register_hook(model): return model.encoder10.register_forward_hook(_hook)

print('Model factory defined.')


Model factory defined.


In [18]:
# ============================ POOLING FUNCTIONS ===============================

def octant_pool(feat_map):
    # feat_map: (1, C, D, H, W) — bottleneck output
    # Returns (8*C,) — 8 octant vectors concatenated
    _, C, D, H, W = feat_map.shape
    dh, hh, wh = D//2, H//2, W//2
    d_sl = [slice(0,dh), slice(dh,D)]
    h_sl = [slice(0,hh), slice(hh,H)]
    w_sl = [slice(0,wh), slice(wh,W)]
    parts = []
    for ds in d_sl:
        for hs in h_sl:
            for ws in w_sl:
                reg = feat_map[:, :, ds, hs, ws]
                parts.append(F.adaptive_avg_pool3d(reg, 1).flatten())
    return torch.cat(parts).cpu().numpy()   # (8*C,)

def mask_pool(feat_map, wt_t, tc_t, et_t):
    # Weighted-average pooling per subregion: background, WT, TC, ET
    # Returns (4*C,)
    _, C, D, H, W = feat_map.shape
    def _pool(mask):
        m = F.interpolate(mask.float().unsqueeze(0).unsqueeze(0),
                          size=(D, H, W), mode='nearest').to(feat_map.device)
        s = m.sum()
        if s < 1: return feat_map.mean(dim=(2,3,4)).flatten()
        return (feat_map * m).sum(dim=(2,3,4)).flatten() / s
    bg_t = (wt_t == 0)
    parts = [_pool(bg_t), _pool(wt_t), _pool(tc_t), _pool(et_t)]
    return torch.cat(parts).cpu().numpy()   # (4*C,)

print('Pooling functions defined.')
print(f'  Octant dim = 8 x {BN_CHANNELS} = {8*BN_CHANNELS}')
print(f'  Mask   dim = 4 x {BN_CHANNELS} = {4*BN_CHANNELS}')


Pooling functions defined.
  Octant dim = 8 x 768 = 6144
  Mask   dim = 4 x 768 = 3072


In [19]:
# ============================ DIM CHECK vs CYPRUS =============================
# CRITICAL: our embedding dim MUST match Cyprus bsf_embeddings_averaged_v2.npz

bsf_emb_path = None
if IS_KAGGLE:
    for p in Path('/kaggle/input').rglob('bsf_embeddings_averaged_v2.npz'):
        bsf_emb_path = str(p); break
else:
    c = Path('/home/moamed/canada_me/explainable_diseas/implementation_cyprus/Phase3/bsf_fold_outputs/embeddings_v2/bsf_embeddings_averaged_v2.npz')
    if c.exists(): bsf_emb_path = str(c)

if bsf_emb_path:
    cy = np.load(bsf_emb_path)
    cy_dim = cy[list(cy.files)[0]].shape[0]
    print(f'Cyprus BSF dim: {cy_dim}')
    if cy_dim == 8 * BN_CHANNELS:
        USE_MASK_POOL = False; FINAL_BSF_DIM = 8 * BN_CHANNELS
        print('-> Octant ONLY (8x768=6144)')
    elif cy_dim == 12 * BN_CHANNELS:
        USE_MASK_POOL = True;  FINAL_BSF_DIM = 12 * BN_CHANNELS
        print('-> Octant + Mask (12x768=9216)')
    else:
        USE_MASK_POOL = True;  FINAL_BSF_DIM = cy_dim
        print(f'WARNING: unexpected dim {cy_dim}')
else:
    USE_MASK_POOL = True; FINAL_BSF_DIM = 12 * BN_CHANNELS
    print(f'Cyprus embeddings not found — default octant+mask: {FINAL_BSF_DIM}')

print(f'Final BSF dim: {FINAL_BSF_DIM}')
print(f'Hybrid dim:    {FINAL_BSF_DIM} + 18 = {FINAL_BSF_DIM + 18}')


Cyprus BSF dim: 8448
Final BSF dim: 8448
Hybrid dim:    8448 + 18 = 8466


In [20]:
# ============================ ROI CROP + SUBREGION MASKS ======================

def roi_crop_resize(vol_4ch, wt_mask):
    # vol_4ch: (4,H,W,D), wt_mask: (H,W,D) bool -> returns (4,96,96,96) tensor
    coords = np.argwhere(wt_mask)
    if len(coords) == 0:
        H, W, D = vol_4ch.shape[1:]
        lo = np.array([max(0,H//2-48), max(0,W//2-48), max(0,D//2-48)])
    else:
        lo = np.maximum(coords.min(0) - ROI_PADDING, 0)
        hi = np.minimum(coords.max(0) + ROI_PADDING + 1, np.array(vol_4ch.shape[1:]))
    if len(coords) == 0:
        hi = lo + 96
    cropped = vol_4ch[:, lo[0]:hi[0], lo[1]:hi[1], lo[2]:hi[2]]
    t = torch.from_numpy(cropped).unsqueeze(0).float()
    t = F.interpolate(t, size=TARGET_SIZE, mode='trilinear', align_corners=False)
    return t.squeeze(0)   # (4,96,96,96)

def build_subregion_masks(mask):
    # mask: (H,W,D) uint8 values {0,1,3}
    # OpenBTAI: NCR=1, ET=3, NO edema
    # WT = any>0,  TC = any>0 (no edema so TC=WT),  ET = ==3
    wt = torch.from_numpy((mask > 0).astype('float32'))
    tc = torch.from_numpy((mask > 0).astype('float32'))  # TC = WT
    et = torch.from_numpy((mask == 3).astype('float32'))
    return wt, tc, et   # each (H,W,D)

print('ROI crop and subregion mask functions defined.')


ROI crop and subregion mask functions defined.


In [21]:
# ============================ SINGLE-SCAN EXTRACTOR ===========================

@torch.no_grad()
def extract_one(model, patient_id, visit_name):
    img_path = PREPROCESS_ROOT / str(patient_id) / visit_name / 'image_t1c.nii.gz'
    if not img_path.exists(): img_path = PREPROCESS_ROOT / str(patient_id) / visit_name / 'image_t1c.nii'
    msk_path = PREPROCESS_ROOT / str(patient_id) / visit_name / 'mask_subregions.nii.gz'
    if not msk_path.exists(): msk_path = PREPROCESS_ROOT / str(patient_id) / visit_name / 'mask_subregions.nii'
    if not img_path.exists() or not msk_path.exists():
        return None
    try:
        img_data = nib.load(str(img_path)).get_fdata().astype('float32')  # (H,W,D)
        msk_data = nib.load(str(msk_path)).get_fdata().astype('uint8')    # (H,W,D)
        # Single channel stored — replicate to 4ch at load time (saves 4x disk space)
        vol_4ch  = np.stack([img_data, img_data, img_data, img_data], axis=0)  # (4,H,W,D)
        wt_mask  = (msk_data > 0)                   # (H,W,D) bool
        vol_crop = roi_crop_resize(vol_4ch, wt_mask) # (4,96,96,96)
        wt_t, tc_t, et_t = build_subregion_masks(msk_data)
        _cache.clear()
        x = vol_crop.unsqueeze(0).to(device)
        with torch.amp.autocast('cuda', enabled=(device.type=='cuda')):
            _ = model(x)
        fm = _cache.get('bn')
        if fm is None: return None
        oct = octant_pool(fm)
        if USE_MASK_POOL:
            msk = mask_pool(fm, wt_t.to(device), tc_t.to(device), et_t.to(device))
            emb = np.concatenate([oct, msk])
        else:
            emb = oct
        return emb.astype('float32')
    except Exception as e:
        print(f'  ERROR {patient_id}/{visit_name}: {e}')
        return None

print('Single-scan extractor defined.')


Single-scan extractor defined.


In [22]:
# ============================ MAIN EXTRACTION LOOP ============================
all_fold_embs = []

for fold in range(3):
    ckpt = CKPT_DIR / f'bsf_fold{fold}_best.pth'
    if not ckpt.exists():
        print(f'WARNING: fold {fold} not found: {ckpt}'); continue

    print(f'{'='*60}')
    print(f'  FOLD {fold} — FROZEN inference (no gradients)')
    print(f'{'='*60}')

    model = create_bsf_model().to(device)
    model = load_checkpoint(model, fold)
    model.eval()
    hook  = register_hook(model)

    fold_embs = {}
    n_ok = n_fail = 0
    t0 = time.time()

    for pid, visit in tqdm(scan_index, desc=f'Fold {fold}'):
        key = f'{pid}__{visit}'
        emb = extract_one(model, pid, visit)
        if emb is not None: fold_embs[key] = emb; n_ok += 1
        else: n_fail += 1

    hook.remove()
    torch.cuda.empty_cache()
    print(f'  Fold {fold}: {n_ok} OK, {n_fail} failed ({time.time()-t0:.0f}s)')

    fo = OUT_DIR / f'openbtai_bsf_fold{fold}_v2.npz'
    np.savez(str(fo), **fold_embs)
    print(f'  Saved: {fo.name}  ({fo.stat().st_size/1e6:.1f} MB)')
    all_fold_embs.append(fold_embs)
    del model

print(f'Extraction complete: {len(all_fold_embs)} folds')


  FOLD 0 — FROZEN inference (no gradients)
  Fold 0: loaded  missing=0 unexpected=0


Fold 0: 100%|██████████| 373/373 [05:26<00:00,  1.14it/s]


  Fold 0: 373 OK, 0 failed (326s)
  Saved: openbtai_bsf_fold0_v2.npz  (13.8 MB)
  FOLD 1 — FROZEN inference (no gradients)
  Fold 1: loaded  missing=0 unexpected=0


Fold 1: 100%|██████████| 373/373 [02:58<00:00,  2.10it/s]


  Fold 1: 373 OK, 0 failed (178s)
  Saved: openbtai_bsf_fold1_v2.npz  (13.8 MB)
  FOLD 2 — FROZEN inference (no gradients)
  Fold 2: loaded  missing=0 unexpected=0


Fold 2: 100%|██████████| 373/373 [02:59<00:00,  2.08it/s]

  Fold 2: 373 OK, 0 failed (180s)
  Saved: openbtai_bsf_fold2_v2.npz  (13.8 MB)
Extraction complete: 3 folds


In [23]:
# ============================ AVERAGE ACROSS FOLDS ============================
all_keys = set()
for fd in all_fold_embs: all_keys.update(fd.keys())

averaged = {}
for key in sorted(all_keys):
    arrs = [fd[key] for fd in all_fold_embs if key in fd]
    if arrs: averaged[key] = np.mean(arrs, axis=0).astype('float32')

print(f'Averaged: {len(averaged)} scans')
if averaged:
    dim   = list(averaged.values())[0].shape[0]
    norms = [np.linalg.norm(v) for v in averaged.values()]
    nanc  = sum(1 for v in averaged.values() if np.any(np.isnan(v)))
    print(f'BSF dim: {dim}')
    print(f'Norms:   [{min(norms):.3f}, {max(norms):.3f}]  std={np.std(norms):.3f}')
    print(f'NaN:     {nanc}')

bsf_out = OUT_DIR / 'openbtai_bsf_embeddings_averaged_v2.npz'
np.savez(str(bsf_out), **averaged)
print(f'Saved: {bsf_out.name}  ({bsf_out.stat().st_size/1e6:.1f} MB)')


Averaged: 373 scans
BSF dim: 9216
Norms:   [50.187, 74.470]  std=1.984
NaN:     0
Saved: openbtai_bsf_embeddings_averaged_v2.npz  (13.8 MB)


## Shape Feature Extraction (No GPU needed)

18-dim shape vector from Cyprus training:
```
[0] log_vol        [1] log_surf       [2] svr
[3] sphericity     [4] elongation     [5] flatness
[6] log_maxdiam    [7] n_labels       [8] t1c_mean*
[9] t1c_std*       [10] t1c_skew*     [11] t1c_kurt*
[12] t1c_ent*      [13] MajorAxis_mm  [14] MinorAxis_mm
[15] LeastAxis_mm  [16] Max2D_mm      [17] GLCM_JointEntropy
```
(*) t1c intensity stats = set to 0 (not in OpenBTAI xlsx)

CRITICAL: apply shape_scaler.pkl fitted on Cyprus — NOT refit.


In [24]:
# ============================ SHAPE FEATURES FROM XLSX ========================
radio_df = pd.read_excel(RADIO_XLSX)
morph_df  = pd.read_excel(MORPH_XLSX)
les_df    = pd.read_csv(PREPROCESS_ROOT / 'openbtai_lesion_selection.csv')
print(f'Radiomics: {radio_df.shape}  Morph: {morph_df.shape}  Lesion: {les_df.shape}')

date_to_visit = {}
for _, row in tl.iterrows():
    date_to_visit[(str(row['patient_id']), str(row['date_str']))] = row['visit_name']

dominant_les = {}
for _, row in les_df.iterrows():
    dominant_les[(str(row['patient_id']), row['visit_name'])] = int(row['dominant_lesion_id'])

shape_dict = {}
for _, row in morph_df.iterrows():
    try:
        pid    = str(int(row['PATIENT']))
        lesion = int(row['LESION'])
        date   = str(int(row['TIME POINT']))
        if len(date) < 8: date = '19' + date.zfill(6)
        visit  = date_to_visit.get((pid, date))
        if visit is None: continue
        if lesion != dominant_les.get((pid, visit), 1): continue
        vol  = max(0.0, float(row.get('TOTALVOLUME', 0)))
        surf = max(0.0, float(row.get('TOTALSURFACE', 0)))
        maxd = max(0.0, float(row.get('MAXDIAMETER3D', 0)))
        v = np.zeros(18, dtype='float32')
        v[0] = float(np.log1p(vol*1000));  v[1] = float(np.log1p(surf*100))
        v[2] = float(surf/(vol+1e-6));     v[6] = float(np.log1p(maxd*10))
        v[7] = 1.0
        shape_dict[f'{pid}__{visit}'] = v
    except (ValueError, TypeError): continue

print(f'Shape dict from morph: {len(shape_dict)} scans')

RADIO_MAP = {
    3: 'original_shape_Sphericity',     4: 'original_shape_Elongation',
    5: 'original_shape_Flatness',       13: 'original_shape_MajorAxisLength',
    14: 'original_shape_MinorAxisLength', 15: 'original_shape_LeastAxisLength',
    16: 'original_shape_Maximum2DDiameterSlice', 17: 'original_glcm_JointEntropy',
}
n_filled = 0
for _, row in radio_df.iterrows():
    try:
        pid    = str(int(row['Patient']))
        date   = str(int(row['Timepoint']))
        if len(date) < 8: date = '19' + date.zfill(6)
        lesion = int(row['Lesion'])
        visit  = date_to_visit.get((pid, date))
        if visit is None: continue
        key = f'{pid}__{visit}'
        if key not in shape_dict: continue
        if lesion != dominant_les.get((pid, visit), 1): continue
        v = shape_dict[key]
        for idx, col in RADIO_MAP.items():
            if col in row.index and not pd.isna(row[col]): v[idx] = float(row[col])
        n_filled += 1
    except (ValueError, TypeError): continue

print(f'Radiomics cols filled: {n_filled}')
print(f'Final shape dict: {len(shape_dict)} scans')


Radiomics: (1599, 1136)  Morph: (596, 17)  Lesion: (143, 5)
Shape dict from morph: 253 scans
Radiomics cols filled: 692
Final shape dict: 253 scans


In [ ]:
# ============================ APPLY SCALER + BUILD HYBRID =====================
# CRITICAL FIX: Only include scans that have REAL shape data from radiomics.
# The 120 scans missing from the xlsx would get zeroed-out shape features,
# which poisons downstream evaluation and TaViT inference. Skip them entirely.

if SCALER_PATH and Path(SCALER_PATH).exists():
    with open(SCALER_PATH, 'rb') as f: shape_scaler = pickle.load(f)
    print(f'Loaded shape_scaler.pkl  type={type(shape_scaler).__name__}')
else:
    shape_scaler = None
    print('WARNING: shape_scaler.pkl not found — appending unscaled shape features')

common   = sorted(set(averaged.keys()) & set(shape_dict.keys()))
bsf_only = sorted(set(averaged.keys()) - set(shape_dict.keys()))
print(f'BSF total         : {len(averaged)}')
print(f'Shape available   : {len(shape_dict)}')
print(f'Common (SAVED)    : {len(common)}')
print(f'No-radiomics SKIP : {len(bsf_only)}')

hybrid_embs = {}
for key in common:
    bsf = averaged[key]
    raw = shape_dict[key].reshape(1, -1)
    shp = shape_scaler.transform(raw)[0] if shape_scaler else raw[0]
    hybrid_embs[key] = np.concatenate([bsf, shp]).astype('float32')

if hybrid_embs:
    dim = list(hybrid_embs.values())[0].shape[0]
    print(f'\nHybrid dim : {dim}  (= {FINAL_BSF_DIM} BSF + 18 shape)')
print(f'Hybrid embeddings: {len(hybrid_embs)}  ')
print(f'Skipped (no radiomics): {len(bsf_only)}')


In [26]:
# ============================ SAVE ALL OUTPUTS ================================
hybrid_out = OUT_DIR / 'openbtai_hybrid_embeddings_v2.npz'
np.savez(str(hybrid_out), **hybrid_embs)
dim = list(hybrid_embs.values())[0].shape[0] if hybrid_embs else 0
print(f'Saved: {hybrid_out.name}  ({hybrid_out.stat().st_size/1e6:.1f} MB)')
print(f'  {len(hybrid_embs)} scans x {dim}-dim')

meta = {
    'dataset':      'OpenBTAI (Ocana-Tienda et al., Scientific Data 2023)',
    'n_scans':      len(hybrid_embs),
    'n_patients':   int(tl['patient_id'].nunique()),
    'bsf_dim':      FINAL_BSF_DIM,
    'shape_dim':    18,
    'hybrid_dim':   FINAL_BSF_DIM + 18,
    'n_folds':      len(all_fold_embs),
    'model_frozen': True,
    'scaler':       'Cyprus training shape_scaler.pkl — not refit',
    'mask_note':    'OpenBTAI {0,1,3}: WT=mask>0, TC=mask>0 (no edema), ET=mask==3',
    't1c_note':     'features 8-12 (t1c stats) set to 0 — not in xlsx',
    'n_bsf_only_skipped': len(bsf_only),
    'note': 'Only scans with real radiomics shape data are saved. BSF-only scans skipped.',
}
with open(OUT_DIR / 'extraction_metadata.json', 'w') as f: json.dump(meta, f, indent=2)
print('Saved: extraction_metadata.json')

print(f'{'='*60}')
print(f'  VALIDATION STEP 2 COMPLETE')
print(f'  {len(hybrid_embs)} hybrid embeddings  dim={FINAL_BSF_DIM+18}')
print(f'{'='*60}')
print('NEXT: Validation_Step3_TaViT_Inference.ipynb')
print('  -> Upload openbtai_embeddings/ to Kaggle')
print('  -> Run FROZEN TaViT on OpenBTAI patient sequences')
print('  -> Extract 128-dim trajectory per patient')


Saved: openbtai_hybrid_embeddings_v2.npz  (13.9 MB)
  373 scans x 9234-dim
Saved: extraction_metadata.json
  VALIDATION STEP 2 COMPLETE
  373 hybrid embeddings  dim=8466
NEXT: Validation_Step3_TaViT_Inference.ipynb
  -> Upload openbtai_embeddings/ to Kaggle
  -> Run FROZEN TaViT on OpenBTAI patient sequences
  -> Extract 128-dim trajectory per patient
